# PCA and clustering

**P1 Core · D3 Synthesis · 110 minutes**

In [ ]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from pathlib import Path

def locate(relative: str, local_name: str | None = None) -> Path:
    candidates = []
    if local_name:
        candidates.append(Path.cwd() / local_name)
    candidates.extend(root / relative for root in [Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Cannot find {relative}. Run from the course clone or place the downloaded data beside this notebook."
    )


In [ ]:
path = locate('datasets/teaching/dry-bean/observations.csv', 'observations.csv')
beans = pd.read_csv(path).groupby('Class', group_keys=False).sample(n=220, random_state=31)
X = beans.drop(columns='Class'); labels = beans['Class']

## Task

Scale features, retain at least 90% PCA variance, fit seven-cluster k-means, and report internal plus post-hoc stability evidence. `Class` cannot enter fitting.

In [ ]:
def pca_cluster(X: pd.DataFrame) -> dict:
    transform = make_pipeline(StandardScaler(), PCA(n_components=.90, random_state=31))
    reduced = transform.fit_transform(X)
    first = KMeans(7, n_init=20, random_state=31).fit_predict(reduced)
    second = KMeans(7, n_init=20, random_state=32).fit_predict(reduced)
    pca = transform.named_steps['pca']
    return {'reduced': reduced, 'first': first, 'second': second,
            'variance': float(pca.explained_variance_ratio_.sum()),
            'silhouette': silhouette_score(reduced, first),
            'seed_stability': adjusted_rand_score(first, second)}

In [ ]:
result = pca_cluster(X)
assert result['variance'] >= .90
assert result['reduced'].shape[1] < X.shape[1]
assert result['seed_stability'] > .85
posthoc_ari = adjusted_rand_score(labels, result['first'])
{'variance': result['variance'], 'silhouette': result['silhouette'],
 'seed_stability': result['seed_stability'], 'class_ari_posthoc': posthoc_ari}

## Transfer

Challenge k-means on anisotropic/noisy/varying-density oracles; compare hierarchical clustering and DBSCAN, bootstrap stability, and domain interpretation.